# Autoresearch: The Complete Interactive Guide

This notebook explains **every AI/ML concept** used in the autoresearch codebase, from zero AI knowledge to expert-level understanding. Every concept includes:
- Intuitive explanations with real-world analogies
- Runnable code examples you can modify and experiment with
- Visualizations and plots
- Deep dives into the custom libraries used

**Prerequisites**: Basic linear algebra (vectors, matrices, dot products). Python knowledge helpful but not required — see Part 0.

---

## Table of Contents

**Part 0 — Python Refresher** *(skip if comfortable with Python)*
- Variables, types, f-strings, data structures
- Functions, classes, decorators, dataclasses
- Common patterns used in this codebase

**Part 1 — Foundations**
1. The Big Picture: What Is This Project?
2. Tensors and PyTorch Basics
3. Neural Networks from Scratch
4. Language Modeling: Predicting the Next Word

**Part 2 — Tokenization**
5. Why Tokenization Matters
6. Byte Pair Encoding (BPE) — The Algorithm
7. Deep Dive: tiktoken and rustbpe Libraries

**Part 3 — The Transformer**
8. Embeddings: Numbers to Vectors
9. Self-Attention: The Core Innovation
10. Multi-Head Attention
11. Positional Encoding (RoPE)
12. Feed-Forward Networks (MLP)
13. Residual Connections & Normalization
14. Sliding Window Attention
15. Value Embeddings (ResFormer)
16. Logit Softcapping
17. The Full GPT Forward Pass

**Part 4 — Training**
18. Loss Functions & Backpropagation
19. Optimizers: AdamW
20. Optimizers: Muon (and the Polar Express)
21. Learning Rate Scheduling
22. Mixed Precision (bfloat16)
23. Gradient Accumulation
24. Weight Initialization
25. Regularization: Weight Decay

**Part 5 — Infrastructure**
26. Flash Attention
27. torch.compile & GPU Optimization
28. Data Pipeline: Shards, Parquet, Dataloading
29. Evaluation: Bits Per Byte (BPB)
30. Performance Metrics: MFU & Throughput

**Appendix**
- A: Research Paper References
- B: Glossary

In [1]:
# Setup: Install visualization dependencies if needed
# Run this cell first!
import subprocess, sys
for pkg in ['matplotlib', 'numpy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('Setup complete!')

Matplotlib is building the font cache; this may take a moment.


Setup complete!


---

# Part 0 — Python Refresher

**Haven't written Python in a while?** This section gets you back up to speed in 5 minutes. Skip it if you're comfortable with Python.

We only cover what you'll actually see in this codebase — no fluff.

---

### Variables & Types

```python
x = 42              # int
lr = 3e-4            # float (scientific notation = 0.0003)
name = "GPT"         # string
training = True      # bool
nothing = None       # null equivalent
```

Python is dynamically typed — no `int x = 42`, just `x = 42`. Variables are just labels pointing to objects.

### F-strings (formatted strings)

```python
loss = 1.847
step = 500
print(f"Step {step}: loss = {loss:.3f}")  # "Step 500: loss = 1.847"
print(f"{step:>6d}")                       # "   500" (right-aligned, 6 chars)
```

You'll see f-strings everywhere in `train.py` for logging.

### Lists, Tuples, Dicts

```python
# Lists — ordered, mutable
layers = [768, 1024, 512]
layers.append(256)          # add to end
first = layers[0]           # 768
last = layers[-1]           # 256
subset = layers[1:3]        # [1024, 512] — slicing is [start:stop)

# Tuples — ordered, immutable (used for shapes, return values)
shape = (batch_size, seq_len, n_embd)
a, b, c = shape             # tuple unpacking

# Dicts — key-value pairs
config = {"n_layer": 12, "n_head": 6, "n_embd": 768}
config["n_layer"]            # 12
config.get("dropout", 0.0)   # 0.0 (default if key missing)
```

### List Comprehensions

```python
# Instead of:
squares = []
for i in range(10):
    squares.append(i ** 2)

# Write:
squares = [i ** 2 for i in range(10)]

# With a filter:
even_squares = [i ** 2 for i in range(10) if i % 2 == 0]
```

You'll see these a lot — they're the Pythonic way to build lists.

### Loops & Enumerate

```python
# Basic loop
for i in range(5):       # 0, 1, 2, 3, 4
    print(i)

# Loop with index AND value (very common in training loops)
for step, batch in enumerate(dataloader):
    if step % 100 == 0:  # every 100 steps
        print(f"Step {step}")
```

### Functions

```python
def compute_loss(logits, targets):
    """Docstring explaining what this does."""
    loss = F.cross_entropy(logits, targets)
    return loss

# Default arguments
def train(lr=3e-4, epochs=10):
    ...

# Call with keyword arguments (order doesn't matter)
train(epochs=5, lr=1e-3)
```

### Classes (you'll see these for the model)

```python
class MLP(nn.Module):
    def __init__(self, n_embd):    # constructor — called when you create an instance
        super().__init__()          # always call parent constructor first
        self.fc = nn.Linear(n_embd, 4 * n_embd)  # self.x stores attributes on the instance

    def forward(self, x):          # called when you do mlp(x)
        return F.relu(self.fc(x))

# Usage:
mlp = MLP(768)         # calls __init__
output = mlp(input)    # calls forward (PyTorch magic via __call__)
```

Key things to know:
- `self` is like `this` in JS/Java — refers to the current instance
- `__init__` is the constructor
- `super().__init__()` calls the parent class constructor
- In PyTorch, `forward()` is called automatically when you call the object like a function

### Decorators

```python
@torch.no_grad()       # disables gradient tracking (saves memory during eval)
def evaluate():
    ...

@dataclass             # auto-generates __init__ from class fields
class Config:
    n_layer: int = 12
    n_head: int = 6
```

A decorator is just a function that wraps another function. `@foo` above `def bar()` is equivalent to `bar = foo(bar)`.

### Dataclasses

```python
from dataclasses import dataclass

@dataclass
class GPTConfig:
    n_layer: int = 12
    n_embd: int = 768

# This auto-generates:
#   def __init__(self, n_layer=12, n_embd=768):
#       self.n_layer = n_layer
#       self.n_embd = n_embd

config = GPTConfig(n_layer=24)  # n_embd defaults to 768
print(config.n_embd)            # 768
```

Used in `train.py` for `GPTConfig` — saves boilerplate.

### Unpacking & Multiple Assignment

```python
# Swap
a, b = b, a

# Unpack function returns
mean, std = torch.std_mean(tensor)

# Star unpacking
first, *rest = [1, 2, 3, 4]  # first=1, rest=[2, 3, 4]
```

### Context Managers (with statements)

```python
# File I/O
with open("data.txt") as f:
    text = f.read()
# File automatically closed after the block

# In PyTorch — very common:
with torch.no_grad():        # temporarily disable gradients
    output = model(x)

with torch.autocast("cuda", torch.bfloat16):  # use mixed precision
    loss = model(x)
```

`with` ensures cleanup happens even if an error occurs.

### Assert & Inline If

```python
# Assert — crash if condition is false (used for sanity checks)
assert len(data) > 0, "Data is empty!"
assert config.n_embd % config.n_head == 0, "Embedding dim must be divisible by heads"

# Ternary / inline if
device = "cuda" if torch.cuda.is_available() else "cpu"
```

### Common Gotchas

| Gotcha | What happens | Fix |
|--------|-------------|-----|
| `a = b = []` | Both point to the SAME list | `a = []; b = []` |
| `def f(x=[]):` | Default list is shared across calls | `def f(x=None): x = x or []` |
| Integer division | `7 / 2 = 3.5` (float!) | `7 // 2 = 3` (integer) |
| `is` vs `==` | `is` checks identity, `==` checks value | Use `==` for values, `is` only for `None` |
| Indentation | Python uses indentation, not braces | Stick to 4 spaces, never mix tabs |

### What You'll See in This Codebase

```python
# Pattern 1: Module registration in __init__, computation in forward
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = Attention(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(x)   # residual connection
        x = x + self.mlp(x)
        return x

# Pattern 2: Dictionary comprehension for parameter groups
param_groups = {name: p for name, p in model.named_parameters() if p.requires_grad}

# Pattern 3: Chained operations
logits = model(tokens)[:, -1, :]  # forward pass, take last position, all vocab

# Pattern 4: In-place operations (underscore suffix)
tensor.mul_(0.5)    # multiply in place (no new tensor created)
tensor.add_(bias)   # add in place
```

You're now ready for the rest of the guide.

---

# Part 1 — Foundations

## 1. The Big Picture: What Is This Project?

**Autoresearch** is a system where an AI coding agent (like Claude) autonomously experiments with training a language model. Think of it like a robot scientist:

1. The robot modifies an experiment (changes model architecture or settings in `train.py`)
2. Runs the experiment (trains for exactly 5 minutes)
3. Measures the result (`val_bpb` — a quality score, lower = better)
4. Keeps improvements, discards regressions
5. Repeats **forever** until stopped

The human writes the "research program" (`program.md`), and the AI executes it. You could go to sleep and wake up to 100 completed experiments.

### What's actually being trained?

A **GPT-style language model** — the same fundamental technology behind ChatGPT, Claude, and every modern LLM. Just at a smaller scale (one GPU, 5-minute runs, ~50M parameters instead of billions).

### The three files that matter

| File | Purpose | Who edits it? |
|------|---------|---------------|
| `prepare.py` | Downloads data, trains tokenizer, provides utilities | **Nobody** (fixed) |
| `train.py` | Model architecture, optimizer, training loop | **The AI agent** |
| `program.md` | Instructions for the AI agent | **The human** |

### The intuition

Imagine you're trying to bake the perfect cake. You have:
- A fixed oven (GPU) and timer (5 minutes)
- A fixed recipe evaluation method (taste test = val_bpb)
- Freedom to change ingredients and techniques (model architecture, hyperparameters)

The AI agent is the baker, trying different recipes and keeping the ones that taste better.

## 2. Tensors and PyTorch Basics

### What is a tensor?

A **tensor** is just a multi-dimensional array of numbers. If you know NumPy arrays, tensors are the same thing but they can run on GPUs.

| Dimensions | Name | Example |
|-----------|------|---------|
| 0 | Scalar | `42.0` — a single number |
| 1 | Vector | `[1.0, 2.0, 3.0]` — a list of numbers |
| 2 | Matrix | A table (rows × columns) |
| 3 | 3D Tensor | A "cube" of numbers — common shape: `[batch, sequence, features]` |
| 4 | 4D Tensor | `[batch, sequence, heads, head_dim]` — used in attention |

### Why PyTorch?

PyTorch is the dominant deep learning framework. It provides:
1. **GPU acceleration**: Operations on tensors run on NVIDIA GPUs (1000x faster than CPU for matrix math)
2. **Automatic differentiation**: Computes gradients automatically (essential for training)
3. **Neural network building blocks**: Pre-built layers, loss functions, optimizers

In [ ]:
# === TENSORS IN ACTION ===
# Let's see what tensors look like at each dimensionality

import numpy as np

# Scalar (0D) — a single number
scalar = np.float32(42.0)
print(f"Scalar: {scalar}, shape: {scalar.shape}, ndim: {scalar.ndim}")

# Vector (1D) — a list of numbers
# In a language model, this could be a single token's embedding
vector = np.array([0.12, -0.45, 0.78, 0.33], dtype=np.float32)
print(f"Vector: {vector}, shape: {vector.shape}, ndim: {vector.ndim}")\

# Matrix (2D) — a table of numbers
# In a language model, this could be a weight matrix for a linear layer
# This one maps from 4-dimensional input to 3-dimensional output
matrix = np.random.randn(4, 3).astype(np.float32)
print(f"\nMatrix (weight matrix):\n{matrix}")
print(f"Shape: {matrix.shape} — maps 4-dim input to 3-dim output")

# 3D Tensor — the standard shape flowing through a transformer
# [batch_size, sequence_length, embedding_dim]
# Think of it as: "128 sequences, each 2048 tokens long, each token is a 512-dim vector"
batch_size, seq_len, embed_dim = 2, 5, 4  # Small example
tensor_3d = np.random.randn(batch_size, seq_len, embed_dim).astype(np.float32)
print(f"\n3D Tensor shape: {tensor_3d.shape}")
print(f"Meaning: {batch_size} sequences, {seq_len} tokens each, {embed_dim}-dim vectors")
print(f"First sequence, first token's vector: {tensor_3d[0, 0, :]}")
print(f"First sequence, second token's vector: {tensor_3d[0, 1, :]}")

# Matrix multiplication — THE fundamental operation in neural networks
# Input vector (4-dim) × Weight matrix (4×3) = Output vector (3-dim)
input_vec = np.array([1.0, 2.0, 3.0, 4.0])
output_vec = input_vec @ matrix  # @ is matrix multiplication
print(f"\nMatrix multiplication (the core of neural networks):")
print(f"Input:  {input_vec} (4-dim)")
print(f"Weight: {matrix.shape} matrix")
print(f"Output: {output_vec} (3-dim)")
print(f"Each output value is a weighted sum of ALL input values (a 'dot product' with one row of the matrix)")

Scalar: 42.0, shape: (), ndim: 0
Vector: [ 0.12 -0.45  0.78  0.33], shape: (4,), ndim: 1
42.0 is a variable

Matrix (weight matrix):
[[ 1.4888678  -0.9869986   3.2633815 ]
 [ 0.73536307 -0.10178693  0.00429599]
 [ 0.7066689   1.361088   -0.1124234 ]
 [ 1.4747626   0.6796583  -1.4214618 ]]
Shape: (4, 3) — maps 4-dim input to 3-dim output

3D Tensor shape: (2, 5, 4)
Meaning: 2 sequences, 5 tokens each, 4-dim vectors
First sequence, first token's vector: [-0.65294534  0.24876444 -0.39491624 -0.70465684]
First sequence, second token's vector: [-0.56090647  0.9616204  -0.38461965  0.80974835]

Matrix multiplication (the core of neural networks):
Input:  [1. 2. 3. 4.] (4-dim)
Weight: (4, 3) matrix
Output: [10.97865087  5.61132482 -2.75114401] (3-dim)
Each output value is a weighted sum of ALL input values (a 'dot product' with one row of the matrix)


## 3. Neural Networks from Scratch

### The core idea

A neural network is a **function with adjustable knobs** (called parameters or weights). You show it examples, measure how wrong it is, then turn the knobs to make it less wrong. Repeat millions of times.

### The building blocks

**Linear layer** (aka fully connected layer, dense layer):
```
output = input × W + b
```
- `W` is a **weight matrix** — the learnable parameters
- `b` is a **bias** (optional, not used in this codebase for most layers)
- This is just matrix multiplication — the most fundamental neural net operation

**Analogy**: A linear layer is like a panel of mixing sliders in a recording studio. Each output channel is a weighted mix of all input channels. The weights are the slider positions.

**Activation functions** (nonlinearities):
Without these, stacking linear layers would just give you another linear function (matrix multiplications compose). Nonlinearities let the network learn complex, curved decision boundaries.

Let's visualize the activation functions used in this codebase:

### Activation Functions: The Complete Guide

Every neural network needs **nonlinear** activation functions. Without them, stacking layers just gives you one big matrix multiply — no matter how deep, the network can only learn straight lines. Activation functions add the "curves" that let networks approximate any function.

---

### How to Choose: The Decision Flowchart

**Step 1: Are you picking for the output layer or a hidden layer?**

If **output layer**, the choice is determined by your task:
- **Regression** (predict a number like temperature, price) → **No activation (linear)**. You want unbounded output.
- **Binary classification** (yes/no, spam/not-spam) → **Sigmoid**. Outputs a probability between 0 and 1.
- **Multi-class classification** (cat/dog/bird) → **Softmax**. Outputs probabilities that sum to 1 across all classes.
- **Multi-label classification** (an image can be both "outdoor" AND "sunny") → **Sigmoid** on each output independently.
- **Bounded output** (e.g., steering angle in [-1, 1]) → **Tanh**.

If **hidden layer**, continue to Step 2.

**Step 2: What architecture are you building?**

| Architecture | Default Choice | Why |
|---|---|---|
| **CNN** (image tasks) | **ReLU** | Fast, proven, good enough for most vision tasks |
| **Transformer / LLM** | **GELU** | Smooth gradients help deep attention networks. Used in GPT, BERT, ViT |
| **Modern LLM (2023+)** | **SwiGLU** | State-of-the-art. Used in LLaMA, PaLM, Gemma, Mistral. Best quality but adds parameters |
| **RNN / LSTM / GRU** | **Tanh** (hidden state) + **Sigmoid** (gates) | Built into the architecture by design |
| **Small/custom model** | **ReLU** or **ReLU²** | Simple, effective. ReLU² adds beneficial sparsity (used in this codebase!) |
| **Diffusion model** | **SiLU (Swish)** | Standard in U-Net architectures for image generation |

**Step 3: Is something going wrong? Troubleshooting guide:**

| Problem | Symptom | Fix |
|---|---|---|
| **Dying neurons** | Many neurons output exactly 0, loss plateaus | Switch ReLU → **Leaky ReLU** or **GELU** |
| **Vanishing gradients** | Deep network barely learns, early layers don't update | Avoid sigmoid/tanh in hidden layers. Use **ReLU/GELU** + residual connections |
| **Exploding activations** | NaN/Inf in forward pass | Add **LayerNorm/BatchNorm**, or use bounded activations (tanh for softcapping) |
| **Model not expressive enough** | Loss plateaus above expected level | Try **SwiGLU** (more parameters, more capacity) or **ReLU²** (sharper feature selection) |
| **Training unstable** | Loss spikes randomly | Try **GELU** (smoother than ReLU). Add normalization before switching activations |

---

### Intuition for Each Activation

**ReLU — "The Bouncer"**
Lets positive values through unchanged, blocks everything negative. Think of a bouncer at a club: you're either in or you're out. This brutal simplicity is why it trains fast — gradients are either 1 (pass through) or 0 (blocked). The downside: neurons can "die" if they always receive negative inputs, permanently outputting 0.

*Use when:* You want something simple that works. CNNs, MLPs, anything where you don't need the fancier options.

**GELU — "The Soft Bouncer"**
Instead of a hard cutoff at zero, GELU uses a smooth, probabilistic curve. Small negative values have a *chance* of getting through. The formula `x * Phi(x)` literally multiplies each value by the probability that a Gaussian random variable would be less than it. This smoothness helps gradient flow in deep transformers.

*Use when:* Building transformers. It's the default in GPT, BERT, and ViT. If you're unsure and working on NLP/transformers, pick this.

**SiLU / Swish — "The Self-Gate"**
`x * sigmoid(x)` — each value gates itself. Large positives pass through, large negatives are suppressed, but small negatives get a slight dip below zero before recovering. This non-monotonicity (the small dip) acts as implicit regularization.

*Use when:* Diffusion models (U-Net), EfficientNet, or when GELU isn't quite cutting it. Also the building block inside SwiGLU.

**Leaky ReLU — "The Forgiving Bouncer"**
Like ReLU, but negative values aren't fully killed — they're just shrunk by a factor (typically 0.01). Gradients always flow, even for negative inputs.

*Use when:* You're using ReLU but noticing dead neurons (activations stuck at 0). A drop-in replacement that fixes the dying ReLU problem.

**Tanh — "The Squasher"**
Squashes any input into [-1, 1]. Zero-centered (unlike sigmoid), so outputs can be both positive and negative. Used inside LSTM/GRU gates and for logit softcapping (preventing attention logits from exploding).

*Use when:* RNNs (built into LSTM/GRU by design), or when you need to bound a value to [-1, 1]. In this codebase, it's used for attention logit softcapping.

**Sigmoid — "The Probability Maker"**
Squashes any input into (0, 1) — perfect for representing probabilities or gates ("how much should I let through?").

*Use when:* Binary classification output, multi-label output, or as a gating mechanism inside attention/LSTMs. Avoid in hidden layers — not zero-centered, vanishing gradients.

**GLU / SwiGLU — "The Learned Gate"**
The most powerful modern activation. The input is split into two paths: one becomes the "content," the other becomes the "gate" (passed through swish/sigmoid). They're multiplied element-wise, so the network dynamically *learns what to let through*. This is why every top LLM uses SwiGLU.

*Use when:* You're building a serious LLM and want state-of-the-art quality. The tradeoff: it requires two weight matrices instead of one in the FFN, so it adds ~50% more parameters to those layers.

**ReLU Squared — "The Sparsity Machine"** *(used in this codebase)*
Squaring after ReLU does two things: (1) small activations get even smaller (0.1 becomes 0.01), creating sharper sparsity, and (2) large activations get amplified. The network becomes more selective about which neurons fire.

*Use when:* You want sharper feature selection than ReLU without the parameter cost of SwiGLU. Research shows ReLU² achieves better loss than standard ReLU in language models. That's exactly why this codebase uses it.

---

### Summary: The Short Version

> **Just tell me what to use:**
> - Building a transformer/LLM? → **GELU** (safe default) or **SwiGLU** (best quality)
> - Building a CNN? → **ReLU**
> - Output layer? → Determined by task (sigmoid/softmax/linear/tanh)
> - Something not working? → Check the troubleshooting table above

In [ ]:
# === ACTIVATION FUNCTIONS: COMPLETE VISUAL REFERENCE ===
# Every activation function explained above, visualized side by side

x = np.linspace(-3, 3, 500)

# --- Define all activation functions ---

# ReLU: max(0, x) — the classic default
relu = np.maximum(0, x)

# GELU: x * Φ(x) — smooth, probabilistic ReLU (used in GPT/BERT)
# Φ(x) is the CDF of standard normal — we approximate it without scipy
# Using the standard approximation: Φ(x) ≈ 0.5 * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x³)))
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

# SiLU (Swish): x * sigmoid(x) — self-gating, non-monotonic
sigmoid_x = 1 / (1 + np.exp(-x))
silu = x * sigmoid_x

# Leaky ReLU: allows small gradient for negatives (alpha=0.01)
leaky_relu = np.where(x > 0, x, 0.01 * x)

# ReLU²: max(0, x)² — sharper sparsity (USED IN THIS CODEBASE)
relu_squared = np.maximum(0, x) ** 2

# Sigmoid: 1/(1+e^(-x)) — squashes to (0, 1)
sigmoid = 1 / (1 + np.exp(-x))

# Tanh: squashes to (-1, 1)
tanh = np.tanh(x)

# --- Plot hidden layer activations (top row) ---
fig, axes = plt.subplots(2, 4, figsize=(22, 9))

hidden_funcs = [
    (relu,         'ReLU\n"The Bouncer"',          '#FF6B6B', 'max(0, x)\nGradient: 1 or 0'),
    (gelu,         'GELU\n"The Soft Bouncer"',      '#9B59B6', 'x·Φ(x)\nSmooth, probabilistic'),
    (silu,         'SiLU / Swish\n"The Self-Gate"',  '#3498DB', 'x·σ(x)\nNote the dip below 0'),
    (leaky_relu,   'Leaky ReLU\n"Forgiving Bouncer"','#E67E22', 'max(0.01x, x)\nNeurons never die'),
]

output_funcs = [
    (relu_squared, 'ReLU²\n"Sparsity Machine"',    '#4ECDC4', 'max(0,x)²\nUsed in THIS model'),
    (sigmoid,      'Sigmoid\n"Probability Maker"',  '#45B7D1', '1/(1+e⁻ˣ)\nOutput: (0, 1)'),
    (tanh,         'Tanh\n"The Squasher"',           '#96CEB4', 'tanh(x)\nOutput: (-1, 1)'),
    (None,         'SwiGLU\n"The Learned Gate"',     '#F39C12', 'gate(x)·content(x)\nNeeds 2 weight matrices'),
]

for i, (func, name, color, desc) in enumerate(hidden_funcs):
    ax = axes[0][i]
    ax.plot(x, func, color=color, linewidth=2.5)
    ax.axhline(y=0, color='gray', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5, alpha=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.text(0.05, 0.95, desc, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7),
            color='white')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(-3, 3)

for i, (func, name, color, desc) in enumerate(output_funcs):
    ax = axes[1][i]
    if func is not None:
        ax.plot(x, func, color=color, linewidth=2.5)
    else:
        # SwiGLU: show conceptual diagram since it needs learned weights
        content = np.tanh(x * 0.8)  # simulated content path
        gate = 1 / (1 + np.exp(-x * 1.5))  # simulated gate path
        swiglu_out = content * gate
        ax.plot(x, content, color='#BDC3C7', linewidth=1.5, linestyle='--', label='content', alpha=0.6)
        ax.plot(x, gate, color='#BDC3C7', linewidth=1.5, linestyle=':', label='gate', alpha=0.6)
        ax.plot(x, swiglu_out, color=color, linewidth=2.5, label='output')
        ax.legend(fontsize=8, loc='upper left')
    ax.axhline(y=0, color='gray', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5, alpha=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.text(0.05, 0.95, desc, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7),
            color='white')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(-3, 3)

axes[0][0].set_ylabel('Output', fontsize=11)
axes[1][0].set_ylabel('Output', fontsize=11)
plt.suptitle('Activation Functions: Complete Visual Reference', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Show gradients (crucial for understanding training behavior) ---
fig2, axes2 = plt.subplots(1, 4, figsize=(22, 3.5))

dx = 0.001
funcs_for_grad = [
    (relu, 'ReLU Gradient', '#FF6B6B', 'Dead zone: gradient=0\nfor all x < 0'),
    (gelu, 'GELU Gradient', '#9B59B6', 'Smooth transition\nno dead zone'),
    (silu, 'SiLU Gradient', '#3498DB', 'Can exceed 1.0!\nNon-monotonic'),
    (relu_squared, 'ReLU² Gradient', '#4ECDC4', 'Gradient grows with x\n(2·max(0,x))'),
]

for i, (func, name, color, desc) in enumerate(funcs_for_grad):
    grad = np.gradient(func, dx)
    axes2[i].plot(x, grad, color=color, linewidth=2)
    axes2[i].axhline(y=0, color='gray', linewidth=0.5, alpha=0.5)
    axes2[i].axvline(x=0, color='gray', linewidth=0.5, alpha=0.5)
    axes2[i].set_title(name, fontsize=11, fontweight='bold')
    axes2[i].text(0.05, 0.95, desc, transform=axes2[i].transAxes, fontsize=8,
                  verticalalignment='top', fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7),
                  color='white')
    axes2[i].grid(True, alpha=0.2)
    axes2[i].set_xlim(-3, 3)

plt.suptitle('Gradients — Why They Matter for Training', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("WHY GRADIENTS MATTER:")
print("During training, the network adjusts weights using gradients (backpropagation).")
print("If the gradient is 0 → the neuron stops learning (dead neuron).")
print("If the gradient is smooth → training is stable and efficient.")
print("This is why GELU/SiLU beat ReLU in transformers — smoother gradients everywhere.")

In [ ]:
# === A TINY NEURAL NETWORK FROM SCRATCH ===
# Let's build a 2-layer network that learns XOR (the simplest non-linear problem)
# XOR: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0
# A single linear layer CANNOT learn this — you need nonlinearity!

np.random.seed(42)

# Training data: XOR truth table
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)

# Initialize weights randomly (2 layers)
# Layer 1: 2 inputs → 4 hidden neurons
W1 = np.random.randn(2, 4).astype(np.float32) * 0.5
b1 = np.zeros((1, 4), dtype=np.float32)
# Layer 2: 4 hidden neurons → 1 output
W2 = np.random.randn(4, 1).astype(np.float32) * 0.5
b2 = np.zeros((1, 1), dtype=np.float32)

learning_rate = 1.0
losses = []

for step in range(2000):
    # ===== FORWARD PASS =====
    # Layer 1: linear + ReLU
    z1 = X @ W1 + b1           # Linear: matrix multiply + bias
    a1 = np.maximum(0, z1)     # ReLU activation

    # Layer 2: linear + sigmoid
    z2 = a1 @ W2 + b2          # Linear
    pred = 1 / (1 + np.exp(-z2))  # Sigmoid (outputs probability 0-1)

    # ===== COMPUTE LOSS =====
    # Binary cross-entropy: measures how wrong our predictions are
    loss = -np.mean(y * np.log(pred + 1e-8) + (1 - y) * np.log(1 - pred + 1e-8))
    losses.append(loss)

    # ===== BACKWARD PASS (backpropagation) =====
    # Compute gradients using the chain rule of calculus
    # This is what PyTorch does automatically with loss.backward()
    d_pred = (pred - y) / len(X)       # Gradient of loss w.r.t. predictions
    d_z2 = d_pred * pred * (1 - pred)  # Through sigmoid
    d_W2 = a1.T @ d_z2                 # Gradient for W2
    d_b2 = d_z2.sum(axis=0, keepdims=True)
    d_a1 = d_z2 @ W2.T                # Propagate back through layer 2
    d_z1 = d_a1 * (z1 > 0)            # Through ReLU (gradient is 0 or 1)
    d_W1 = X.T @ d_z1                 # Gradient for W1
    d_b1 = d_z1.sum(axis=0, keepdims=True)

    # ===== OPTIMIZER STEP =====
    # Subtract gradients (move parameters in direction that reduces loss)
    W1 -= learning_rate * d_W1
    b1 -= learning_rate * d_b1
    W2 -= learning_rate * d_W2
    b2 -= learning_rate * d_b2

# Show results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss over time
ax1.plot(losses, color='#FF6B6B', linewidth=2)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss (lower = better)')
ax1.set_title('Training Loss Over Time', fontweight='bold')
ax1.grid(True, alpha=0.2)
ax1.annotate('Network learns XOR!', xy=(500, losses[500]), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='white'), xytext=(800, 0.5), color='white')

# Plot 2: Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
z1_grid = np.maximum(0, grid @ W1 + b1)
z2_grid = z1_grid @ W2 + b2
pred_grid = 1 / (1 + np.exp(-z2_grid))
ax2.contourf(xx, yy, pred_grid.reshape(xx.shape), levels=50, cmap='RdYlBu_r', alpha=0.8)
ax2.scatter(X[:, 0], X[:, 1], c=y.flatten(), cmap='RdYlBu_r', s=200,
           edgecolors='white', linewidths=2, zorder=5)
for i, (xi, yi) in enumerate(X):
    ax2.annotate(f'XOR={int(y[i,0])}', (xi+0.05, yi+0.08), fontsize=11, color='white', fontweight='bold')
ax2.set_title('Learned Decision Boundary', fontweight='bold')
ax2.set_xlabel('Input 1')
ax2.set_ylabel('Input 2')

plt.tight_layout()
plt.show()

print("Final predictions (should be close to [0, 1, 1, 0]):")
for i in range(4):
    print(f"  Input: {X[i]} → Predicted: {pred[i,0]:.4f} (Target: {y[i,0]:.0f})")

print("\nThis is EXACTLY what happens in autoresearch, just at massive scale:")
print("  - Instead of 4 examples, we have millions of text sequences")
print("  - Instead of 10 parameters, we have ~50 million")
print("  - Instead of predicting XOR, we predict the next token in text")

## 4. Language Modeling: Predicting the Next Word

### The task

A language model is trained on one deceptively simple task: **given a sequence of tokens, predict the next one.**

```
Input:    "The cat sat on the"
Target:   "mat"
```

At every position, the model outputs a **probability distribution** over the entire vocabulary (8,192 scores). The training signal says "the correct answer was token #4521" and the model adjusts to make that token more probable next time.

### Why this simple task produces intelligence

To predict well, the model must learn:
- **Grammar**: "The cat *sat*" not "The cat *purple*"
- **Semantics**: "The doctor prescribed *medicine*" not "*furniture*"  
- **World knowledge**: "The capital of France is *Paris*"
- **Reasoning**: "If x = 3 and y = x + 2, then y = *5*"
- **Style/tone**: A formal text continues formally, a casual text casually

### Causal (autoregressive) modeling

**"Causal"** means each token can only see tokens **before** it, never future tokens. This is critical because:
1. During text **generation**, future tokens don't exist yet
2. During **training**, it lets us get 2047 training examples from a single 2048-token sequence (predict each token from all preceding ones)

**Analogy**: It's like reading a book with a sliding window — you can see everything you've read so far, but the rest is covered up. You're trying to guess the next word before uncovering it.

### Softmax: turning scores into probabilities

The model outputs raw scores (called **logits**) for each vocabulary token. To interpret these as probabilities, we apply **softmax**:

$$P(token_i) = \frac{e^{logit_i}}{\sum_j e^{logit_j}}$$

This ensures all probabilities are positive and sum to 1. Higher logits → higher probabilities.

In [ ]:
# === LANGUAGE MODELING VISUALIZED ===
# Let's see how the model makes predictions at each position

# Simulated example: the model sees "The cat sat" and predicts next tokens
# (In reality these would be token IDs, but we'll use words for clarity)

tokens = ["The", "cat", "sat", "on", "the", "mat"]

# Simulated model confidence at each position (in a real model, these come from softmax)
# The model predicts a probability distribution over the ENTIRE vocabulary at each position
fig, ax = plt.subplots(figsize=(16, 6))

# For each position, show what the model "sees" and what it predicts
for i in range(len(tokens) - 1):
    context = tokens[:i+1]
    target = tokens[i+1]

    # Draw context (what model sees)
    for j, tok in enumerate(context):
        ax.text(i * 2.5 + 0.1, 4 - j * 0.5, tok,
               fontsize=12, color='#4ECDC4',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='#2C3E50', edgecolor='#4ECDC4'))

    # Draw prediction arrow
    ax.annotate('', xy=(i * 2.5 + 0.5, 4 - len(context) * 0.5 - 0.3),
               xytext=(i * 2.5 + 0.5, 4 - len(context) * 0.5 + 0.1),
               arrowprops=dict(arrowstyle='->', color='#FF6B6B', lw=2))

    # Draw target
    ax.text(i * 2.5 + 0.1, 4 - len(context) * 0.5 - 0.7, f'→ "{target}"',
           fontsize=12, color='#FF6B6B', fontweight='bold')

    # Position label
    ax.text(i * 2.5 + 0.5, 5, f'Position {i}', fontsize=10, ha='center', color='gray')

ax.set_xlim(-0.5, 13)
ax.set_ylim(-0.5, 5.5)
ax.set_title('Causal Language Modeling: Predict Next Token from Context',
            fontsize=14, fontweight='bold')
ax.text(0, -0.3, 'Each position can only see tokens BEFORE it (never future tokens)',
       fontsize=11, color='gray', style='italic')
ax.axis('off')
plt.tight_layout()
plt.show()

# === SOFTMAX VISUALIZATION ===
# Show how raw scores (logits) become probabilities

logits = np.array([2.5, 1.0, 0.5, -1.0, -2.0, 0.8, 3.1, 0.1])
vocab = ["mat", "floor", "chair", "dog", "car", "rug", "ground", "table"]

# Apply softmax
exp_logits = np.exp(logits - logits.max())  # subtract max for numerical stability
probs = exp_logits / exp_logits.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Raw logits
colors = ['#FF6B6B' if p == probs.max() else '#4ECDC4' for p in probs]
ax1.barh(vocab, logits, color=colors, alpha=0.8)
ax1.set_xlabel('Logit (raw score)')
ax1.set_title('Raw Model Output (Logits)', fontweight='bold')
ax1.axvline(x=0, color='gray', linewidth=0.5)
ax1.grid(True, alpha=0.2, axis='x')

# After softmax
ax2.barh(vocab, probs, color=colors, alpha=0.8)
ax2.set_xlabel('Probability')
ax2.set_title('After Softmax (Probabilities)', fontweight='bold')
for i, (v, p) in enumerate(zip(vocab, probs)):
    ax2.text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=10, color='white')
ax2.grid(True, alpha=0.2, axis='x')

plt.suptitle('Context: "The cat sat on the ___"', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('The model assigns a probability to EVERY token in the vocabulary (8,192 tokens).')
print('Here we show 8 for clarity. The correct answer "mat" should get the highest probability.')

---

# Part 2 — Tokenization

## 5. Why Tokenization Matters

Language models work with **numbers**, not text. We need a systematic way to convert text ↔ numbers. This conversion is called **tokenization**.

### Three approaches (and why BPE wins)

| Approach | Vocab Size | Tokens for "unhappiness" | Pros | Cons |
|----------|-----------|------------------------|------|------|
| **Character-level** | ~256 | `u,n,h,a,p,p,i,n,e,s,s` (11 tokens) | Can handle any text | Very long sequences, slow |
| **Word-level** | ~500,000+ | `unhappiness` (1 token, if in vocab) | Short sequences | Huge vocab, can't handle new words |
| **BPE (subword)** | 8,192 | `un,happi,ness` (3 tokens) | Best of both worlds | Slightly complex |

### The key trade-off

- **Smaller vocab** (this codebase: 8,192): Smaller embedding table, but more tokens per document
- **Larger vocab** (GPT-4: ~100,000): Fewer tokens per document, but larger embedding table

**Analogy**: Think of tokenization like Morse code. Common letters (E, T) get short codes (·, −). Rare letters (Q, Z) get long codes (−−·−, −−··). BPE does this for text: common words get single tokens, rare words get broken into pieces.

## 6. Byte Pair Encoding (BPE) — The Algorithm

BPE was originally a data compression algorithm (1994). Its adaptation for NLP tokenization has become the universal standard.

### How BPE training works (step by step)

In [ ]:
# === BPE ALGORITHM FROM SCRATCH ===
# Let's implement BPE step-by-step so you can see exactly how it works

from collections import Counter

def train_bpe(text, num_merges):
    """Train a BPE tokenizer from scratch.

    This is a simplified version of what rustbpe does in prepare.py.
    The real implementation is in Rust for speed, but the algorithm is identical.
    """
    # Step 1: Start with individual characters as our initial "tokens"
    # In real BPE, we start with bytes (256 possible values)
    tokens = list(text)
    vocab = sorted(set(tokens))
    merges = []  # Record of all merges we make
    history = [list(tokens)]  # Save state at each step for visualization

    print(f"Initial vocabulary ({len(vocab)} tokens): {vocab[:20]}...")
    print(f"Initial sequence length: {len(tokens)} tokens")
    print(f"Initial text: {''.join(tokens[:60])}...")
    print()

    for i in range(num_merges):
        # Step 2: Count all adjacent pairs
        pairs = Counter()
        for j in range(len(tokens) - 1):
            pair = (tokens[j], tokens[j + 1])
            pairs[pair] += 1

        if not pairs:
            break

        # Step 3: Find the most frequent pair
        best_pair = pairs.most_common(1)[0]
        pair, count = best_pair

        # Step 4: Merge that pair into a new token
        new_token = pair[0] + pair[1]
        merges.append((pair, new_token, count))
        vocab.append(new_token)

        # Replace all occurrences of the pair with the new token
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and tokens[j] == pair[0] and tokens[j + 1] == pair[1]:
                new_tokens.append(new_token)
                j += 2  # Skip both tokens in the pair
            else:
                new_tokens.append(tokens[j])
                j += 1
        tokens = new_tokens
        history.append(list(tokens))

        print(f"Merge {i+1}: '{pair[0]}' + '{pair[1]}' → '{new_token}' "
              f"(appeared {count} times, sequence now {len(tokens)} tokens)")

    return vocab, merges, history


# Train on a small example
text = "the cat sat on the mat the cat ate the fat rat that sat on the mat"
print("=" * 70)
print(f"Training text: \"{text}\"")
print("=" * 70)
print()

vocab, merges, history = train_bpe(text, num_merges=12)

print(f"\nFinal vocabulary size: {len(vocab)} tokens")
print(f"Final sequence length: {len(history[-1])} tokens (was {len(history[0])})")
print(f"Compression ratio: {len(history[0]) / len(history[-1]):.1f}x")

In [ ]:
# === VISUALIZE BPE COMPRESSION ===
# Watch how the sequence gets shorter as merges are applied

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8))

# Plot 1: Sequence length over merges
lengths = [len(h) for h in history]
ax1.plot(range(len(lengths)), lengths, 'o-', color='#4ECDC4', linewidth=2, markersize=8)
for i, (m, l) in enumerate(zip(merges, lengths[1:])):
    ax1.annotate(f"'{m[1]}'", (i+1, l), textcoords="offset points",
                xytext=(0, 12), ha='center', fontsize=8, color='white', rotation=30)
ax1.set_xlabel('Number of Merges Applied')
ax1.set_ylabel('Sequence Length (tokens)')
ax1.set_title('BPE Compression: Sequence Gets Shorter with Each Merge', fontweight='bold')
ax1.grid(True, alpha=0.2)

# Plot 2: Show the actual tokens at a few key stages
stages = [0, 3, 6, len(history)-1]
colors = ['#FF6B6B', '#FFD93D', '#4ECDC4', '#96CEB4']
for row, (stage_idx, color) in enumerate(zip(stages, colors)):
    toks = history[stage_idx]
    display_toks = toks[:30]  # Show first 30 tokens
    label = f"After {stage_idx} merges ({len(toks)} tokens)"
    ax2.text(-0.5, row, label, fontsize=10, va='center', ha='right', color=color, fontweight='bold')
    for j, tok in enumerate(display_toks):
        ax2.text(j * 0.9, row, f'[{tok}]', fontsize=7, va='center', ha='center',
                color=color, alpha=0.9,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', edgecolor=color, alpha=0.6))

ax2.set_xlim(-8, 28)
ax2.set_ylim(-0.8, len(stages) - 0.2)
ax2.set_title('Token Sequence at Different Stages of BPE', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

print("\nKEY INSIGHT: BPE is essentially a learned compression scheme.")
print("Common patterns ('the', 'at', 'th') become single tokens.")
print("Rare patterns stay as individual characters/bytes.")

## 7. Deep Dive: tiktoken and rustbpe Libraries

### The two tokenizer libraries in this codebase

The codebase uses **two** tokenizer libraries for different purposes:

| Library | Language | Purpose in this codebase | Speed |
|---------|----------|--------------------------|-------|
| **rustbpe** | Rust (Python bindings) | **Training** the tokenizer (learning the merges) | Very fast |
| **tiktoken** | Rust (Python bindings) | **Using** the tokenizer at runtime (encoding/decoding) | Very fast |

**Why two?** Training a tokenizer (finding optimal merges) is a different problem than using one (applying merges to new text). `rustbpe` specializes in training; `tiktoken` specializes in fast runtime encoding.

### How tiktoken works internally

tiktoken (by OpenAI) stores the tokenizer as a dictionary called **mergeable_ranks**:
```python
{
    b'h': 0,        # rank 0 (lowest = merged first among base tokens)
    b'e': 1,
    b't': 2,
    ...
    b'th': 256,     # merged token (rank 256)
    b'the': 512,    # merged token (rank 512)
    ...
}
```

When encoding text, tiktoken:
1. Pre-splits the text using a regex pattern (the `SPLIT_PATTERN`)
2. For each chunk, applies BPE merges in rank order (lowest rank = highest priority)
3. Returns the resulting token IDs

### The split pattern

Before BPE merges, text is pre-split using this regex:
```
'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+
```

This ensures merges don't cross word boundaries in problematic ways:
- Contractions stay together: `don't` → `don` + `'t`
- Numbers split every 2 digits: `12345` → `12` + `34` + `5`
- Whitespace is handled intelligently

### Special tokens

Four tokens with special meaning that never appear in normal text:
```python
SPECIAL_TOKENS = ["<|reserved_0|>", "<|reserved_1|>", "<|reserved_2|>", "<|reserved_3|>"]
```

`<|reserved_0|>` is used as **BOS (Beginning Of Sequence)** — prepended to every document to signal "new document starts here."

### The token_bytes lookup table

For evaluation (BPB metric), we need to know how many UTF-8 bytes each token represents:
```python
"the"  → 3 bytes
"é"    → 2 bytes (in UTF-8)
"你"   → 3 bytes (in UTF-8)
"<|reserved_0|>" → 0 bytes (special token, not real text)
```

This is precomputed once and saved as a PyTorch tensor for fast lookup during evaluation.

### How prepare.py uses these libraries

```
1. rustbpe.Tokenizer().train_from_iterator(text, vocab_size, pattern)
   → Trains BPE: reads text, finds optimal merges, builds vocabulary

2. tiktoken.Encoding(mergeable_ranks=..., special_tokens=...)
   → Creates a fast runtime tokenizer from the trained merges

3. pickle.dump(enc) → saves to disk for train.py to load later
```

---

# Part 3 — The Transformer Architecture

## 8. Embeddings: Numbers to Vectors

### The problem

Token IDs are arbitrary integers (e.g., "the" = 1234, "cat" = 5678). These numbers have no meaningful relationship — 1234 isn't "closer" to 1235 in any linguistic sense.

### The solution: embedding tables

An **embedding** is a learnable lookup table that maps each integer ID to a dense vector:

```
token_id 1234 ("the")  →  [0.12, -0.45, 0.78, ..., 0.33]  (512 numbers)
token_id 5678 ("cat")  →  [-0.56, 0.23, 0.11, ..., -0.89]  (512 numbers)
```

The dimensionality (512 in default config, called `n_embd`) is a hyperparameter.

**Analogy**: Imagine describing every person with a 512-number "fingerprint" that captures their personality, skills, appearance, etc. Similar people would have similar fingerprints. Embeddings do this for words — similar words end up with similar vectors.

### Why it works

During training, the model adjusts these vectors so that:
- **Similar words** end up nearby in vector space
- **Relationships** are encoded as directions (famously: king - man + woman ≈ queen)
- The vectors capture **meaning**, not just spelling

In [ ]:
# === EMBEDDINGS VISUALIZED ===
# Let's create a small embedding table and see how it works

np.random.seed(42)

# Simulate a small vocabulary with 2D embeddings (so we can plot them)
vocab_words = ["king", "queen", "man", "woman", "prince", "princess",
               "dog", "cat", "puppy", "kitten", "car", "truck", "bike"]

# Manually create embeddings that show meaningful relationships
embeddings_2d = {
    "king":     [3.0, 4.0],
    "queen":    [3.0, 2.0],
    "man":      [2.0, 3.8],
    "woman":    [2.0, 1.8],
    "prince":   [3.5, 3.5],
    "princess": [3.5, 1.5],
    "dog":      [-2.0, 3.0],
    "cat":      [-2.0, 1.5],
    "puppy":    [-1.5, 3.2],
    "kitten":   [-1.5, 1.7],
    "car":      [-3.0, -2.0],
    "truck":    [-2.5, -2.5],
    "bike":     [-3.5, -1.5],
}

fig, ax = plt.subplots(figsize=(12, 8))

# Color by category
categories = {
    "Royalty": (["king", "queen", "prince", "princess"], '#FF6B6B'),
    "People": (["man", "woman"], '#FFD93D'),
    "Animals": (["dog", "cat", "puppy", "kitten"], '#4ECDC4'),
    "Vehicles": (["car", "truck", "bike"], '#96CEB4'),
}

for cat_name, (words, color) in categories.items():
    xs = [embeddings_2d[w][0] for w in words]
    ys = [embeddings_2d[w][1] for w in words]
    ax.scatter(xs, ys, c=color, s=150, label=cat_name, edgecolors='white', linewidths=1.5, zorder=5)
    for w in words:
        ax.annotate(w, (embeddings_2d[w][0], embeddings_2d[w][1]),
                   textcoords="offset points", xytext=(8, 8), fontsize=11, color='white')

# Draw the famous analogy: king - man + woman ≈ queen
king = np.array(embeddings_2d["king"])
man = np.array(embeddings_2d["man"])
woman = np.array(embeddings_2d["woman"])
result = king - man + woman  # Should be near "queen"

ax.annotate('', xy=woman[:2], xytext=man[:2],
           arrowprops=dict(arrowstyle='->', color='#FFD93D', lw=2, ls='--'))
ax.annotate('', xy=result[:2], xytext=king[:2],
           arrowprops=dict(arrowstyle='->', color='#FFD93D', lw=2, ls='--'))
ax.text(-0.5, 4.5, 'king - man + woman ≈ queen', fontsize=13,
       color='#FFD93D', fontweight='bold',
       bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1a2e', edgecolor='#FFD93D'))

ax.set_xlabel('Embedding Dimension 1', fontsize=12)
ax.set_ylabel('Embedding Dimension 2', fontsize=12)
ax.set_title('Word Embeddings in 2D Space\n(Real embeddings have 512 dimensions, shown in 2D for visualization)',
            fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("In this codebase:")
print(f"  - Embedding table shape: [vocab_size=8192, n_embd=512]")
print(f"  - Total embedding parameters: {8192 * 512:,} = {8192 * 512 / 1e6:.1f}M")
print(f"  - Each token is represented as a 512-dimensional vector")
print(f"  - nn.Embedding is just a matrix where row i = embedding for token i")

## 9. Self-Attention: The Core Innovation

Self-attention is **the** key mechanism that makes transformers work. Everything else is relatively standard neural network building blocks.

### The problem attention solves

Consider: *"The **animal** didn't cross the street because **it** was too tired."*

What does "it" refer to? "animal." To understand this, the model processing "it" needs to "look back" at "animal" and gather its information. **That's what attention does.**

### The Query-Key-Value framework

**Analogy**: Think of attention like a library search system:

- **Query (Q)**: Your search query — "What am I looking for?" Each token asks this.
- **Key (K)**: Book titles/tags — "What do I contain?" Each token advertises this.
- **Value (V)**: The actual book content — "Here's my information." Each token offers this.

The process:
1. Each token broadcasts its **Key** ("I'm about animal/verb/noun...")
2. Each token sends out a **Query** ("I need to find who 'it' refers to...")
3. Queries are matched against Keys (dot product → similarity score)
4. High-scoring tokens contribute their **Values** to the output

### The math (step by step)

```
1. Project each token into Q, K, V vectors:
   Q = x @ W_q    K = x @ W_k    V = x @ W_v

2. Compute attention scores (how relevant is each key to each query):
   scores = Q @ K^T / sqrt(head_dim)

3. Apply causal mask (can't attend to future tokens):
   scores[future_positions] = -infinity

4. Convert to probabilities:
   weights = softmax(scores)

5. Gather information:
   output = weights @ V
```

In [ ]:
# === SELF-ATTENTION FROM SCRATCH ===
# Let's compute attention step-by-step on a tiny example

np.random.seed(42)

tokens = ["The", "animal", "didn't", "cross", "it"]
seq_len = len(tokens)
embed_dim = 8   # Small dimension so we can see the numbers
head_dim = 8

# Step 1: Create random embeddings for each token
x = np.random.randn(seq_len, embed_dim).astype(np.float32) * 0.5

# Step 2: Create Q, K, V projection matrices (these are the learnable parameters)
W_q = np.random.randn(embed_dim, head_dim).astype(np.float32) * 0.3
W_k = np.random.randn(embed_dim, head_dim).astype(np.float32) * 0.3
W_v = np.random.randn(embed_dim, head_dim).astype(np.float32) * 0.3

# Step 3: Project embeddings into Q, K, V
Q = x @ W_q  # [5, 8] — each token's "what am I looking for?"
K = x @ W_k  # [5, 8] — each token's "what do I contain?"
V = x @ W_v  # [5, 8] — each token's "what info do I provide?"

print("=" * 70)
print("SELF-ATTENTION STEP BY STEP")
print("=" * 70)
print(f"\nTokens: {tokens}")
print(f"Embedding dim: {embed_dim}, Head dim: {head_dim}")
print(f"\nQ shape: {Q.shape} — queries for each token")
print(f"K shape: {K.shape} — keys for each token")
print(f"V shape: {V.shape} — values for each token")

# Step 4: Compute attention scores
# scores[i][j] = "how much should token i attend to token j?"
scores = Q @ K.T / np.sqrt(head_dim)
print(f"\nRaw attention scores (Q @ K^T / sqrt(d)):")
print(f"Shape: {scores.shape} — score for every (query, key) pair")

# Step 5: Apply CAUSAL MASK
# Token i can only attend to tokens 0..i (not future tokens)
mask = np.triu(np.ones((seq_len, seq_len)), k=1)  # Upper triangle = future
scores_masked = scores.copy()
scores_masked[mask == 1] = -1e9  # Set future positions to -infinity

print(f"\nCausal mask (1 = blocked, 0 = allowed):")
for i, tok in enumerate(tokens):
    row = ["." if mask[i][j] == 0 else "X" for j in range(seq_len)]
    print(f"  {tok:8s} can see: {' '.join(row)}  ({tokens[:i+1]})")

# Step 6: Softmax → attention weights (probabilities)
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

weights = softmax(scores_masked)

# Step 7: Weighted sum of values
output = weights @ V  # [5, 8] — each token's new representation

# === VISUALIZE THE ATTENTION PATTERN ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Attention weights heatmap
im1 = ax1.imshow(weights, cmap='YlOrRd', aspect='auto')
ax1.set_xticks(range(seq_len))
ax1.set_yticks(range(seq_len))
ax1.set_xticklabels(tokens, fontsize=11)
ax1.set_yticklabels(tokens, fontsize=11)
ax1.set_xlabel('Attending TO (Keys)', fontsize=12)
ax1.set_ylabel('Attending FROM (Queries)', fontsize=12)
ax1.set_title('Attention Weights\n(brighter = more attention)', fontweight='bold')
for i in range(seq_len):
    for j in range(seq_len):
        color = 'black' if weights[i, j] > 0.3 else 'white'
        ax1.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')
plt.colorbar(im1, ax=ax1, shrink=0.8)

# Plot 2: Causal mask visualization
mask_visual = np.where(mask == 1, np.nan, scores)
im2 = ax2.imshow(mask_visual, cmap='coolwarm', aspect='auto')
ax2.set_xticks(range(seq_len))
ax2.set_yticks(range(seq_len))
ax2.set_xticklabels(tokens, fontsize=11)
ax2.set_yticklabels(tokens, fontsize=11)
ax2.set_xlabel('Key position', fontsize=12)
ax2.set_ylabel('Query position', fontsize=12)
ax2.set_title('Causal Mask Effect\n(gray = blocked/future)', fontweight='bold')
# Mark blocked positions
for i in range(seq_len):
    for j in range(seq_len):
        if mask[i][j] == 1:
            ax2.text(j, i, '✗', ha='center', va='center', fontsize=14, color='gray')
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

print("\nKEY TAKEAWAY: Each token's output is a weighted average of all VISIBLE tokens' values.")
print("The weights are determined by query-key similarity (how relevant each token is).")
print("The causal mask ensures we only look backward, never at future tokens.")

## 10. Multi-Head Attention

### Why multiple heads?

A single attention head can only focus on **one type of relationship** at a time. Multi-head attention runs several attention computations **in parallel**, each learning different patterns:

- **Head 1**: Syntax (subject-verb agreement)
- **Head 2**: Coreference (what does "it" refer to?)
- **Head 3**: Semantic relationships (cause-effect)
- **Head 4**: Local context (nearby words)

**Analogy**: It's like having multiple people read the same document, each with different colored highlighters for different purposes. One highlights grammar, another highlights character references, another highlights plot points. Together they capture more than any one reader could.

### How it works

Instead of one big attention with `head_dim = n_embd`, we split into `n_head` smaller attentions:

```
n_embd = 512, n_head = 4 → head_dim = 128

Head 1: Q₁(128), K₁(128), V₁(128) → output₁(128)
Head 2: Q₂(128), K₂(128), V₂(128) → output₂(128)
Head 3: Q₃(128), K₃(128), V₃(128) → output₃(128)
Head 4: Q₄(128), K₄(128), V₄(128) → output₄(128)

Concatenate: [output₁ | output₂ | output₃ | output₄] = 512
Final projection (W_out): 512 → 512  (mixes across heads)
```

Total compute is the same as single-head, but each head learns different patterns.

### Grouped Query Attention (GQA)

A memory optimization: multiple query heads share the same K, V heads. In this codebase, `n_head = n_kv_head` (standard MHA), but the code supports GQA if the agent wants to try it.

## 11. Positional Encoding: RoPE (Rotary Position Embeddings)

### The problem

Self-attention treats its input as a **set** — it doesn't know token order! "Dog bites man" and "Man bites dog" would produce the same attention patterns without positional information.

### RoPE: Encoding position through rotation

RoPE encodes position by **rotating** each token's Q and K vectors by an angle proportional to position. Different vector dimensions rotate at different frequencies (like the hands of a clock — hour hand moves slowly, second hand moves fast).

**Analogy**: Imagine each token wearing a unique "position badge" made of rotating arrows. When two tokens compute their attention score (dot product), the result naturally depends on how far apart they are — nearby tokens' arrows are nearly aligned, distant tokens' arrows point in very different directions.

### Key properties

1. **Relative position**: The attention score between tokens depends only on their **distance**, not absolute positions. Position 100→95 gives the same result as 10→5.
2. **Multi-frequency**: Low-frequency rotations capture coarse position (nearby vs. far). High-frequency rotations capture exact distance.
3. **Extrapolation**: Can handle sequences longer than seen during training (to some degree).

In [ ]:
# === ROTARY POSITION EMBEDDINGS (RoPE) VISUALIZED ===
# RoPE rotates vectors by angles that depend on position

head_dim = 8
base = 10000
seq_len = 64

# Compute the rotation frequencies for each dimension pair
# Lower dimensions → higher frequency (fast rotation)
# Higher dimensions → lower frequency (slow rotation)
channel_range = np.arange(0, head_dim, 2, dtype=np.float32)
inv_freq = 1.0 / (base ** (channel_range / head_dim))

positions = np.arange(seq_len, dtype=np.float32)
# Outer product: each position × each frequency = rotation angles
angles = np.outer(positions, inv_freq)

cos_vals = np.cos(angles)
sin_vals = np.sin(angles)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Rotation angles for different dimension pairs
ax = axes[0, 0]
for i, (freq, label) in enumerate(zip(inv_freq, [f'Dim pair {j}-{j+1}' for j in range(0, head_dim, 2)])):
    ax.plot(positions, angles[:, i], linewidth=2, label=f'{label} (freq={freq:.4f})')
ax.set_xlabel('Token Position')
ax.set_ylabel('Rotation Angle (radians)')
ax.set_title('RoPE Rotation Angles by Dimension Pair', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Plot 2: Cos and Sin for the first dimension pair
ax = axes[0, 1]
ax.plot(positions, cos_vals[:, 0], color='#4ECDC4', linewidth=2, label='cos (dim 0-1)')
ax.plot(positions, sin_vals[:, 0], color='#FF6B6B', linewidth=2, label='sin (dim 0-1)')
ax.plot(positions, cos_vals[:, -1], color='#4ECDC4', linewidth=2, linestyle='--', label='cos (dim 6-7)')
ax.plot(positions, sin_vals[:, -1], color='#FF6B6B', linewidth=2, linestyle='--', label='sin (dim 6-7)')
ax.set_xlabel('Token Position')
ax.set_ylabel('Value')
ax.set_title('Cos/Sin Values (fast freq = solid, slow freq = dashed)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Plot 3: How rotation affects dot product (relative position encoding)
# Show that Q·K depends on DISTANCE between positions, not absolute positions
ax = axes[1, 0]
# Create a random query vector and apply RoPE at different positions
q_base = np.random.randn(head_dim).astype(np.float32) * 0.5
k_base = np.random.randn(head_dim).astype(np.float32) * 0.5

def apply_rope(vec, pos, cos_v, sin_v):
    d = len(vec) // 2
    x1, x2 = vec[:d], vec[d:]
    c, s = cos_v[pos], sin_v[pos]
    y1 = x1 * c + x2 * s
    y2 = -x1 * s + x2 * c
    return np.concatenate([y1, y2])

# Compute dot product for different (query_pos, key_pos) pairs
distances = range(-30, 31)
dot_products_pos10 = []
dot_products_pos40 = []
for dist in distances:
    key_pos_a = 10 + dist
    key_pos_b = 40 + dist
    if 0 <= key_pos_a < seq_len:
        q_rot = apply_rope(q_base, 10, cos_vals, sin_vals)
        k_rot = apply_rope(k_base, key_pos_a, cos_vals, sin_vals)
        dot_products_pos10.append(np.dot(q_rot, k_rot))
    else:
        dot_products_pos10.append(np.nan)
    if 0 <= key_pos_b < seq_len:
        q_rot = apply_rope(q_base, 40, cos_vals, sin_vals)
        k_rot = apply_rope(k_base, key_pos_b, cos_vals, sin_vals)
        dot_products_pos40.append(np.dot(q_rot, k_rot))
    else:
        dot_products_pos40.append(np.nan)

ax.plot(distances, dot_products_pos10, color='#4ECDC4', linewidth=2, label='Query at pos 10')
ax.plot(distances, dot_products_pos40, color='#FF6B6B', linewidth=2, linestyle='--', label='Query at pos 40')
ax.set_xlabel('Distance (key position - query position)')
ax.set_ylabel('Q·K Dot Product')
ax.set_title('RoPE Makes Attention Depend on DISTANCE\n(both curves nearly identical!)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)
ax.axvline(x=0, color='gray', linewidth=0.5)

# Plot 4: Visual of the rotation in 2D
ax = axes[1, 1]
# Show how a single 2D vector gets rotated at different positions
vec_2d = np.array([1.0, 0.0])
positions_to_show = [0, 4, 8, 16, 32]
colors_rot = ['#FF6B6B', '#FFD93D', '#4ECDC4', '#96CEB4', '#45B7D1']
for pos, color in zip(positions_to_show, colors_rot):
    angle = angles[pos, 0]  # First dimension pair
    rotated = np.array([vec_2d[0] * np.cos(angle) - vec_2d[1] * np.sin(angle),
                       vec_2d[0] * np.sin(angle) + vec_2d[1] * np.cos(angle)])
    ax.annotate('', xy=rotated, xytext=(0, 0),
               arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(rotated[0]*1.15, rotated[1]*1.15, f'pos={pos}', fontsize=10,
           color=color, fontweight='bold', ha='center')

circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--', alpha=0.3)
ax.add_patch(circle)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('RoPE: Same Vector Rotated\nby Different Positions', fontweight='bold')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("RoPE in the code (train.py):")
print("  - _precompute_rotary_embeddings(): Precomputes cos/sin for all positions")
print("  - apply_rotary_emb(): Applies rotation to Q and K vectors")
print("  - The cos/sin are stored as buffers (not learnable parameters)")

## 12. Feed-Forward Networks (MLP)

Each transformer block has two halves: attention (tokens talk to each other) and **MLP** (each token processed independently). The MLP acts as the model's "memory bank" — it stores and retrieves factual knowledge.

### Architecture: Expand → Activate → Compress

```
input (512-dim) → expand 4x (2048-dim) → ReluSquared → compress (512-dim)
```

**Analogy**: Think of it like decompressing a ZIP file, processing the contents, and re-compressing. The 4x expansion gives a temporary "workspace" with more room to represent complex patterns.

### ReluSquared: Why square the ReLU?

The activation `ReLU(x)² = max(0, x)²` has two nice properties:
1. **Sparsity** from ReLU: ~50% of neurons are zero → computational efficiency
2. **Amplification** from squaring: strong signals get amplified, weak signals stay small → sharper feature detection

## 13. Residual Connections & Normalization

### The vanishing gradient problem

In a deep network (8 layers), gradients get multiplied together during backpropagation. If each layer attenuates the gradient by 0.9x:
- After 8 layers: 0.9⁸ = 0.43 (manageable)
- After 50 layers: 0.9⁵⁰ = 0.005 (barely any signal reaches early layers!)

### Residual connections: the fix

Instead of `x = f(x)`, do `x = x + f(x)`:
```python
x = x + self.attn(norm(x), ...)   # Attention + skip connection
x = x + self.mlp(norm(x))          # MLP + skip connection
```

The `+ x` creates a "highway" for gradients to flow directly through, bypassing the layers. Even if the attention/MLP gradients vanish, the gradient can still flow through the skip connection.

**Analogy**: It's like a road with exits. Even if the exits (layers) are congested, traffic can still flow on the main highway (the residual connection).

### Enhanced residual: x0 shortcut

This codebase adds an extra shortcut back to the **initial embedding**:
```python
x = resid_lambda[i] * x + x0_lambda[i] * x0
```

This prevents the original token identity from being "washed out" by many layers of processing. Deep layers can still access "what token am I?" directly.

### RMS Normalization

**RMSNorm(x) = x / sqrt(mean(x²))**

Keeps vector magnitudes stable across layers. Applied **before** each sublayer ("pre-norm" style). Without normalization, values can grow exponentially through layers, causing numerical instability.

## 14. Sliding Window Attention

Full attention costs O(T²) — every token attends to every other token. But most relevant context is **nearby**. Sliding window attention limits each token to the nearest W tokens.

The pattern `"SSSL"` repeats across layers:
- **S** (Short): Attend to nearest half-context (1024 tokens)
- **L** (Long): Attend to full context (2048 tokens)
- Last layer always uses Long

This saves ~75% of attention compute while preserving long-range capabilities through the Long layers.

## 15. Value Embeddings (ResFormer)

**Problem**: In deep transformers, the original token identity gets "washed out." By layer 8, the hidden state for "cat" may not retain much about being "cat."

**Solution**: A separate embedding table provides a **direct path** from token IDs to the attention value vectors, bypassing the hidden state entirely:

```python
v_normal = W_v @ hidden_state          # Standard V from hidden state
v_direct = ValueEmbed[token_id]         # Direct V from token ID
v = v_normal + gate * v_direct          # Mix with learned gate
```

Applied to alternating layers to save memory. The gate starts at 1.0 (neutral) and is learned.

## 16. Logit Softcapping

The model's output logits can become very large, causing numerical instability. **Softcapping** smoothly limits them:

```python
logits = 15 * tanh(logits / 15)
```

- Small logits (|x| << 15): Pass through unchanged
- Large logits (|x| >> 15): Clipped to ±15

This prevents extreme overconfidence without creating hard discontinuities in the gradient.

## 17. The Full GPT Forward Pass

Here's the complete data flow:

In [ ]:
# === THE FULL GPT FORWARD PASS — ARCHITECTURE DIAGRAM ===

fig, ax = plt.subplots(figsize=(18, 14))
ax.set_xlim(0, 18)
ax.set_ylim(0, 14)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color, fontsize=10, subtext=None):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                     facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.12 if subtext else 0), text,
           ha='center', va='center', fontsize=fontsize, color='white', fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.2, subtext,
               ha='center', va='center', fontsize=8, color='white', alpha=0.7)

def draw_arrow(ax, x1, y1, x2, y2, color='white'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Title
ax.text(9, 13.5, 'GPT Forward Pass (as implemented in train.py)', ha='center',
       fontsize=16, fontweight='bold', color='white')

# Step 1: Input
draw_box(ax, 7, 12.3, 4, 0.7, 'Token IDs', '#2C3E50', subtext='[batch=128, seq=2048] ints')

# Step 2: Token Embedding
draw_arrow(ax, 9, 12.3, 9, 11.9)
draw_box(ax, 6.5, 11.2, 5, 0.7, 'Token Embedding (wte)', '#E74C3C', subtext='Lookup table: token_id → 512-dim vector')

# Step 3: RMS Norm + Save x0
draw_arrow(ax, 9, 11.2, 9, 10.8)
draw_box(ax, 7, 10.1, 4, 0.7, 'RMS Norm + Save x0', '#9B59B6', subtext='Normalize, save for x0 shortcut')

# Transformer blocks
draw_arrow(ax, 9, 10.1, 9, 9.7)

# Block i (expanded)
block_y = 5.0
block_h = 4.5
rect = mpatches.FancyBboxPatch((3.5, block_y), 11, block_h, boxstyle="round,pad=0.2",
                                 facecolor='#1a1a2e', edgecolor='#F39C12', linewidth=2, alpha=0.5)
ax.add_patch(rect)
ax.text(14.2, block_y + block_h - 0.3, '× 8 layers', fontsize=12,
       color='#F39C12', fontweight='bold')

# Enhanced residual
draw_box(ax, 5, 8.5, 8, 0.6, 'Enhanced Residual: x = λ_resid·x + λ_x0·x0', '#F39C12', fontsize=9)
draw_arrow(ax, 9, 9.7, 9, 9.1)

# Attention sub-block
draw_arrow(ax, 9, 8.5, 9, 8.1)
draw_box(ax, 4, 7.3, 4.5, 0.7, 'Self-Attention', '#3498DB',
        subtext='Q,K,V + RoPE + Flash Attn3')

# Value embedding (side)
draw_box(ax, 10, 7.3, 3.5, 0.7, 'Value Embed', '#1ABC9C', subtext='Direct token→V path')
draw_arrow(ax, 11.75, 7.3, 8, 7.65)

# Attention output + residual
draw_arrow(ax, 6.25, 7.3, 6.25, 6.9)
draw_box(ax, 5, 6.2, 3, 0.6, 'x = x + attn_out', '#2C3E50', fontsize=9)

# MLP
draw_arrow(ax, 6.5, 6.2, 6.5, 5.8)
draw_box(ax, 4, 5.1, 5, 0.7, 'MLP: expand 4x → ReluSquared → compress', '#27AE60', fontsize=9)

# MLP output + residual
draw_arrow(ax, 9.5, 6.5, 9.5, 6.1)
draw_box(ax, 9.5, 5.7, 3, 0.3, 'x = x + mlp_out', '#2C3E50', fontsize=8)

# After blocks
draw_arrow(ax, 9, 5.0, 9, 4.6)
draw_box(ax, 7, 3.9, 4, 0.7, 'Final RMS Norm', '#9B59B6')

# LM Head
draw_arrow(ax, 9, 3.9, 9, 3.5)
draw_box(ax, 6, 2.8, 6, 0.7, 'LM Head: 512 → 8192', '#E74C3C',
        subtext='Project to vocabulary size')

# Softcap
draw_arrow(ax, 9, 2.8, 9, 2.4)
draw_box(ax, 6.5, 1.7, 5, 0.7, 'Softcap: 15·tanh(logits/15)', '#F39C12',
        subtext='Limit logits to [-15, +15]')

# Loss
draw_arrow(ax, 9, 1.7, 9, 1.3)
draw_box(ax, 6, 0.5, 6, 0.7, 'Cross-Entropy Loss', '#E74C3C',
        subtext='Compare predictions to actual next tokens')

# Side annotations
ax.text(1.5, 7.5, 'Each block\nrefines the\ntoken vectors\nthrough\nattention\n(communication)\nand MLP\n(computation)',
       fontsize=9, color='gray', ha='center', va='center', style='italic')

ax.text(16, 7.5, 'Sliding window:\nSSSL pattern\n\nS = 1024 tokens\nL = 2048 tokens\n(full context)',
       fontsize=9, color='gray', ha='center', va='center', style='italic')

plt.tight_layout()
plt.show()

print("Total parameters in default config (depth=8, n_embd=512):")
print("  Token embedding:    8192 × 512 = 4.2M")
print("  8 transformer blocks:          ≈ 38M")
print("  Value embeddings (4 layers):   ≈ 6.7M")
print("  LM head:            512 × 8192 = 4.2M")
print("  Per-layer scalars:             ≈ 16")
print("  ──────────────────────────────────")
print("  Total:                         ≈ 50M parameters")